### Display MTEB results
MTEB results are saved in .json files, one for each task. 
This notebook aggregates the results for multiple tasks and multiple models.

In [ ]:
import os
os.getcwd()

'/gpfs01/berens/user/lholzwarth'

In [2]:
import mteb
import json
import pandas as pd

2024-11-27 17:15:39.783743: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-11-27 17:15:39.798585: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-11-27 17:15:39.803097: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-27 17:15:39.817712: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-11-27 17:15:40.960661: W tensorflow/compiler/tf2

In [ ]:
# get task type 
mteb.get_task("ArguAna").metadata.type

'Retrieval'

In [4]:
data = {}
main_dir = "/gpfs01/berens/user/lholzwarth/text_embedding/MTEB/sparse_results"

# Traverse folders and subfolders
for (root, dirs, files) in os.walk(main_dir):
    # Identify model version from the folder structure
    model_version = os.path.basename(root)
    print(model_version)
        
    for file in files:
        # Skip unwanted files
        if file in {"model_meta.json"} or not file.endswith('.json'):
            continue
            
        file_path = os.path.join(root, file)
        try:
            # Read JSON and extract necessary fields
            with open(file_path, 'r') as f:
                json_data = json.load(f)
                task_name = json_data.get("task_name")
                main_score = json_data.get("scores", {}).get("test", {})[0]["main_score"]
                main_score = round(main_score*100, 2)
                    
                if task_name and main_score is not None:
                    if task_name not in data:
                        data[task_name] = {}
                    data[task_name][model_version] = main_score
        except (json.JSONDecodeError, KeyError):
            print(f"Error parsing file: {file_path}")


sparse_results
Tfidf
tfidf_log
tfidf_rnd100_log
1.0
Error parsing file: /gpfs01/berens/user/lholzwarth/text_embedding/MTEB/sparse_results/Tfidf/1.0/BibleNLPBitextMining.json
svd_log_old
svd
svd_log
tfidf_rnd768_log
sentence-transformers__average_word_embeddings_glove.6B.300d
no_revision_available


In [ ]:
df = pd.DataFrame.from_dict(data, orient='index')
df.index.name = "task_name"
df["task_types"] = [mteb.get_task(task).metadata.type for task in df.index]
df

,tfidf_log,tfidf_rnd100_log,1.0,svd_log_old,svd_log,tfidf_rnd768_log,svd,no_revision_available,task_types
task_name,,,,,,,,,
StackExchangeClusteringP2P,17.53,17.87,18.80,34.01,34.12,17.00,NaN,NaN,Clustering
STSBenchmark,68.57,66.46,69.23,NaN,44.72,68.44,NaN,NaN,STS
ArxivClusteringP2P,35.28,8.32,26.72,41.73,40.23,29.12,NaN,NaN,Clustering
BiorxivClusteringP2P,26.44,4.00,20.56,33.80,33.74,18.14,NaN,NaN,Clustering
STS15,73.43,70.74,74.01,NaN,52.51,73.06,NaN,NaN,STS
SciDocsRR,62.32,52.08,62.30,NaN,52.71,61.72,NaN,NaN,Reranking
SCIDOCS,14.69,3.83,13.31,NaN,5.31,12.57,NaN,NaN,Retrieval
MedrxivClusteringP2P,20.82,11.16,19.30,30.00,30.11,17.13,NaN,NaN,Clustering
MindSmallReranking,23.45,24.75,23.44,NaN,26.63,27.00,NaN,NaN,Reranking


In [6]:
task_selection = ["ArguAna", "ArxivClusteringP2P", "BiorxivClusteringP2P", "MedrxivClusteringP2P", "MindSmallReranking",
                 "RedditClusteringP2P", "SCIDOCS", "SciDocsRR", "StackExchangeClusteringP2P", "STS15", "STS16",
                 "STSBenchmark"]

df.loc[task_selection].sort_values("task_types")

,tfidf_log,tfidf_rnd100_log,1.0,svd_log_old,svd_log,tfidf_rnd768_log,svd,no_revision_available,task_types
task_name,,,,,,,,,
ArxivClusteringP2P,35.28,8.32,26.72,41.73,40.23,29.12,NaN,NaN,Clustering
BiorxivClusteringP2P,26.44,4.00,20.56,33.80,33.74,18.14,NaN,NaN,Clustering
MedrxivClusteringP2P,20.82,11.16,19.30,30.00,30.11,17.13,NaN,NaN,Clustering
RedditClusteringP2P,40.34,11.37,31.54,45.96,34.59,32.42,NaN,NaN,Clustering
StackExchangeClusteringP2P,17.53,17.87,18.80,34.01,34.12,17.00,NaN,NaN,Clustering
MindSmallReranking,23.45,24.75,23.44,NaN,26.63,27.00,NaN,NaN,Reranking
SciDocsRR,62.32,52.08,62.30,NaN,52.71,61.72,NaN,NaN,Reranking
ArguAna,52.54,11.63,42.48,NaN,41.35,43.38,NaN,NaN,Retrieval
SCIDOCS,14.69,3.83,13.31,NaN,5.31,12.57,NaN,NaN,Retrieval
